# Regularization and Overfitting

Implement L2 regularization and dropout in NumPy.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)


## 1. L2 Regularization
Compare training with and without L2 penalty.

In [ ]:
X = np.random.randn(50, 3)
true_w = np.array([2.0, 0.0, 0.0])
y = X @ true_w + 0.1 * np.random.randn(50)

def train(lam, epochs=300, lr=0.01):
    w = np.zeros(3)
    losses = []
    for _ in range(epochs):
        pred = X @ w
        loss = np.mean((pred - y)**2) + (lam/2)*np.sum(w**2)
        grad = 2 * X.T @ (pred - y) / len(y) + lam * w
        w -= lr * grad
        losses.append(loss)
    return w, losses

w_no, losses_no = train(lam=0.0)
w_reg, losses_reg = train(lam=0.5)

plt.plot(losses_no, label='No reg'); plt.plot(losses_reg, label='L2 lam=0.5')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('L2 Regularization Effect')
plt.legend(); plt.show()
print('No reg weights:', np.round(w_no, 3))
print('L2 reg weights:', np.round(w_reg, 3))


## 2. Dropout

In [ ]:
def dropout(a, p=0.5, training=True):
    """Inverted dropout: scale by 1/(1-p) during training to preserve expected value."""
    if not training:
        return a
    mask = (np.random.rand(*a.shape) > p).astype(float)
    return (a * mask) / (1 - p)

np.random.seed(0)
a = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
train_out = dropout(a, p=0.5, training=True)
eval_out = dropout(a, p=0.5, training=False)
print('Original:  ', a)
print('Train drop:', train_out)
print('Eval drop: ', eval_out, '  (unchanged)')
print('Expected sum preserved:', round(a.mean(), 2), 'vs dropped mean:', round(train_out[train_out>0].mean(), 2))


## 3. Simulate overfitting with train vs val curves

In [ ]:
np.random.seed(42)
# Simulate overfitting curves
epochs = np.arange(1, 51)
train_loss = 1.0 * np.exp(-epochs * 0.06) + 0.02
val_loss = 0.8 * np.exp(-epochs * 0.04) + 0.02 * epochs * 0.01 + 0.15
val_loss = np.minimum(val_loss, 0.8)  # cap

plt.plot(epochs, train_loss, label='Train loss')
plt.plot(epochs, val_loss, label='Validation loss', linestyle='--')
plt.axvline(np.argmin(val_loss)+1, color='red', linestyle=':', label='Early stop')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Overfitting: Train vs Validation')
plt.legend(); plt.show()


## 4. Exercise: implement early stopping
Stop training when validation loss hasn't improved for 10 epochs.

In [ ]:
def early_stopping_demo(patience=10):
    best_val = float('inf')
    no_improve = 0
    best_epoch = 0
    for epoch, vl in enumerate(val_loss):
        if vl < best_val:
            best_val = vl
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f'Early stop at epoch {epoch+1}, best epoch {best_epoch+1}, best val loss {best_val:.4f}')
            return
    print('Training complete')

early_stopping_demo(patience=10)
